In [ ]:
import os
import sys
import pandas as pd


sys.path.append(os.path.abspath(".."))

from utils.config import Configuration, load_config

config = load_config("../config.yaml")

tsv_dir = os.path.join("..", config.nsddata_responses_tsv_dir)

In [ ]:
subject_id = 1

tsv_path = os.path.join(
        tsv_dir,
        "ppdata",
        f"subj{subject_id:02d}",
        "behav",
        "responses.tsv",
    )

    # having all of the array loaded at once results in memory pressure
    # reduce to needed subset!

tsv_data = pd.read_csv(tsv_path, sep="\t")

In [ ]:
tsv_data

In [ ]:
from collections import Counter
import numpy as np
from collections import defaultdict



results = []

sample_dict = defaultdict(list)

for i in range(1, 40+1):
    result_dict_session = {}

    entries_session = tsv_data[tsv_data["SESSION"] == i]
    entry_ids = entries_session["73KID"].tolist()

    for entry_id in list(set(entry_ids)):
        entries_sample = entries_session[entries_session["73KID"] == entry_id]


        result_dict_session[entry_id] = {
            "count" : len(entries_sample) 
        }

        for row_i, row in entries_sample.iterrows():
            sample_dict[entry_id].append({
                "session" : int(row["SESSION"]),
                "index" : row_i,
                "session_index" : row_i - (int(row["SESSION"])-1)*750,
                "73KID" : entry_id
            }) 

            result_dict_session[entry_id]["session"] = int(row["SESSION"])
            result_dict_session[entry_id]["index"] = row_i
            result_dict_session[entry_id]["session_index"] = row_i - (int(row["SESSION"])-1)*750


    if i==1:
        print(result_dict_session)

    results.append(result_dict_session)


res_matrix = np.zeros((40, 40))
for i in range(40):
    entry_ids_i = list(results[i].keys())
    for j in range(40):
        if i==j:
            continue

        entry_ids_j = list(results[j].keys())

        overlap = list(set(entry_ids_i) & set(entry_ids_j))

        res_matrix[i,j] = len(overlap)
        res_matrix[j,i] = len(overlap)




In [ ]:
import seaborn as sns
sns.set_theme(rc={'figure.figsize':(14.7,11.27)})
sns.heatmap(res_matrix, annot=True)

In [ ]:
dict(sample_dict)

In [ ]:
import h5py as h5

subject = 1


sessions = [1, 10, 20, 30, 40]

freesurfer_dir = config.freesurfer_dir
v1_rois_lh = os.path.join(freesurfer_dir, f"subj{subject:02d}", "label", "customrois", f"lh.subj{subject:02d}.testrois.mgz")
v1_rois_rh = os.path.join(freesurfer_dir, f"subj{subject:02d}", "label", "customrois", f"rh.subj{subject:02d}.testrois.mgz")



import nibabel as nib

# Load MGZ files
lh_img = nib.load(v1_rois_lh)
rh_img = nib.load(v1_rois_rh)

# Get the data arrays
lh = lh_img.get_fdata()
rh = rh_img.get_fdata()

# 'V1v': 1, 'V1d': 2
v1_rois = np.concatenate((np.squeeze(lh), np.squeeze(rh)))

v1_indices = np.where((v1_rois == 1) | (v1_rois == 2))[0]

In [ ]:

loaded_betas = []

for session in sessions:
    betas_dir = os.path.join(config.nsddata_betas_dir, "ppdata", f"subj{subject:02d}", "nativesurface", "betas_fithrf_GLMdenoise_RR")
    beta_lh = os.path.join(betas_dir, f"lh.betas_session{session:02d}.hdf5")
    beta_rh = os.path.join(betas_dir, f"rh.betas_session{session:02d}.hdf5")



    with h5.File(beta_lh, 'r') as f:
        betas_lh = f['betas'][:]  # Shape: [n_samples, n_voxels]


    with h5.File(beta_rh, 'r') as f:
        betas_rh = f['betas'][:]  # Shape: [n_samples, n_voxels]

    betas_concat = np.concatenate((betas_lh, betas_rh), axis=1)


    loaded_betas.append(betas_concat)


In [ ]:
for i in range(len(sessions)):
    print(f"session: {sessions[i]} - {np.mean(loaded_betas[i])}")
    print(f"session: {sessions[i]} - {np.mean(loaded_betas[i][:, v1_indices])}")




In [ ]:
# 18040: 2
from IPython.display import Image, display
import scipy.stats
import matplotlib.pyplot as plt
import seaborn as sns


# ids_to_loop = [46003, 18040]
# ids_to_loop = list(sample_dict.keys())[:5]
ids_to_loop = [18170+1, 34200+1, 16649+1]
plots = []

for id_loop in ids_to_loop:
    betas = []
    print(f"Looping {id_loop=}")
    print(sample_dict[id_loop])

    print(list(set([e["session"] for e in sample_dict[id_loop]])))

    session_numbers = [e["session"] for e in sample_dict[id_loop]]
    for session in list(set([e["session"] for e in sample_dict[id_loop]])):
        betas_dir = os.path.join(config.nsddata_betas_dir, "ppdata", f"subj{subject:02d}", "nativesurface", "betas_fithrf_GLMdenoise_RR")
        beta_lh = os.path.join(betas_dir, f"lh.betas_session{session:02d}.hdf5")
        beta_rh = os.path.join(betas_dir, f"rh.betas_session{session:02d}.hdf5")


        with h5.File(beta_lh, 'r') as f:
            betas_lh = f['betas'][:]  # Shape: [n_samples, n_voxels]


        with h5.File(beta_rh, 'r') as f:
            betas_rh = f['betas'][:]  # Shape: [n_samples, n_voxels]

        betas_concat = np.concatenate((betas_lh, betas_rh), axis=1)

        for sample in sample_dict[id_loop]:
            if sample["session"] == session:
                print(f"Taking sample {sample['session_index']} from {session=} {np.mean(betas_concat[sample['session_index']])=}")
                betas.append(betas_concat[sample["session_index"]])

    
    corrs = np.zeros((len(betas), len(betas)))
    for i in range(len(betas)):
        for j in range(len(betas)):
            corr = scipy.stats.pearsonr(betas[i], betas[j])
            corrs[i, j] = corr.statistic

    plt.figure(figsize=(8, 6))
    ax = sns.heatmap(
        corrs, 
        annot=True, 
        fmt=".1f", 
        cbar=True, 
        xticklabels=session_numbers,  # Set x-axis labels
        yticklabels=session_numbers   # Set y-axis labels

    )
    plt.xlabel("Session Number")
    plt.ylabel("Session Number")
    plt.title(f"Correlation Heatmap for {id_loop}")

    # Store the figure instead of the seaborn object
    plots.append(plt.gcf())  
    plt.close()  # Close the figure to prevent overlapping in Jupyter Notebook

# Display all figures
for plot in plots:
    display(plot)


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

print(np.max(betas[0]))
print(np.min(betas[0]))


stacked = np.vstack(betas)
plt.figure(figsize=(12, 2))  # Wide figure for better visualization
# plt.imshow(betas[0].reshape(1, -1), cmap="coolwarm", aspect="auto", vmax=500, vmin=-500)

plt.imshow(stacked, cmap="coolwarm", aspect="auto", vmax=500, vmin=-500)
plt.colorbar(label="Value")
plt.xlabel("Index")
plt.yticks([1, 2, 3])  # Remove y-axis labels (since it's a single row)
plt.title("1D Array Visualization")
plt.show()


In [ ]:
from sklearn.preprocessing import MinMaxScaler

rescaled = stacked.copy()
print(rescaled.shape)

def standardize(arr):
    mean = np.mean(arr)
    std = np.std(arr)

    z_scores = (arr - mean) / std

    return z_scores

for i in range(rescaled.shape[0]):
    rescaled[i] = standardize(rescaled[i])

fig, axes = plt.subplots(len(rescaled), 1, figsize=(12, 6), sharex=True, sharey=True)



# Plot each array as a separate heatmap
for i, ax in enumerate(axes):
    im = ax.imshow(rescaled[i].reshape(1, -1), cmap="coolwarm", aspect="auto", vmin=-2, vmax=2)
    ax.set_yticks([])  # Remove y-ticks
    ax.set_ylabel(f"Array {i+1}")  # Label each row

# Colorbar (one for all plots)
fig.colorbar(im, ax=axes, orientation="vertical", label="Value")

plt.xlabel("Index")
plt.suptitle("Comparison of 3 Arrays with Shared Colormap")
plt.show()

In [ ]:
corrs = np.zeros((len(betas), len(betas)))
for i in range(len(betas)):
    for j in range(len(betas)):
        corr = scipy.stats.pearsonr(stacked[i], stacked[j])
        corrs[i, j] = corr.statistic


print(corrs)

plt.figure(figsize=(8, 6))
ax = sns.heatmap(
    corrs, 
    annot=True, 
    fmt=".1f", 
    cbar=True, 
    xticklabels=session_numbers,  # Set x-axis labels
    yticklabels=session_numbers   # Set y-axis labels

)
plt.xlabel("Session Number")
plt.ylabel("Session Number")
plt.title(f"Correlation Heatmap for {id_loop}")




In [ ]:
def optimal_scaling_factor(sample1, sample2):
    return np.sum(sample1 * sample2) / np.sum(sample1 ** 2)



f_opt = optimal_scaling_factor(rescaled[0], rescaled[1])